`01_study_ds.ipynb`: en el notebook actual se procederá a estudiar el dataset descargado procedente de [AnimalTrack: A Benchmark for Multi-Animal Tracking in the Wild](https://hengfan2010.github.io/projects/AnimalTrack/download.html) del cual solo se filtró por tres clases: "horse", "pig" and "penguin".

# Importaciones

In [6]:
from dataset.study_ds import get_dataset_paths, extract_video_info, summary_df
from utils.utils import load_video, load_gt, format_gt
import os

# Estudio del dataset

El dataset se encuentra dentro de la carpeta `dataset` y hay una carpeta por cada animal seleccionado. Dentro de cada carpeta hay dos subcarpetas: `videos` y `gt`. 
- `videos`: contiene los videos de los animales en formato ".mp4".
- `gt`: contiene los ground truth de los videos en formato ".txt".

Cada video con nombre "ejemplo.mp4" tiene un ground truth con nombre "ejemplo_gt.txt". Por otro lado en el dataset se informa del formato de las etiquetas, cada línea del archivo .txt tiene el siguiente formato:

`[frame_id, target_id, top_left_x, top_left_y, width, confidence, class, visibility]`

## Importación del dataset

Se ha definido la función `get_ds_paths` del cual se obtiene la ruta del dataset y posteriormente por cada una de las clases se crean pares de rutas relacionando las rutas de los videos y de los ground truths.

In [2]:
ds_info = get_dataset_paths()

In [3]:
print(f'Ruta del dataset: {ds_info["dataset_path"]}')
print(f"Number of labels: {len(ds_info['labels'])}")
print("Labels:")
for label in ds_info['labels']:
    print(f"- {label}")

Ruta del dataset: /root/ObjectRecognition/dataset
Number of labels: 3
Labels:
- horse
- pig
- penguin


Ejemplo de una ruta de videos y gt

In [4]:
example = ds_info['data']['horse'][0]
example['video']

'/root/ObjectRecognition/dataset/horse/videos/horse_1.mp4'

Extraemos el número de vídeos que tiene cada animal.

In [5]:
for animal in ds_info['labels']:
    print(f"Número de vídeos para {animal}: {len(ds_info['data'][animal])}")

Número de vídeos para horse: 7
Número de vídeos para pig: 5
Número de vídeos para penguin: 6


Hay que contestar una serie de preguntas acerca del dataset para saber si es homogeneo o no, y sus características:
* ¿Todos los vídeos tienen la misma resolución?
* ¿Tienen el mismo número de frames?
* ¿Cuantas etiquetas hay en cada uno de los frames?

In [8]:
df_summary = summary_df(ds_info)

In [18]:
df_videos_tech = df_summary.groupby('video_name')[['width', 'height', 'fps', 'total_frames']].first()
df_videos_tech['duration_sec'] = df_videos_tech['total_frames'] / df_videos_tech['fps']
display(df_videos_tech)

,width,height,fps,total_frames,duration_sec
video_name,,,,,
horse_1,2560,1440,30.0,284,9.466667
horse_2,2560,1440,30.0,310,10.333333
horse_3,2560,1440,30.0,1434,47.800000
horse_4,2560,1440,30.0,206,6.866667
horse_5,2560,1440,30.0,325,10.833333
horse_6,2560,1440,30.0,315,10.500000
horse_7,2560,1440,30.0,1315,43.833333
penguin_1,2560,1440,30.0,314,10.466667
penguin_2,2560,1440,30.0,325,10.833333


In [3]:
def extract_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return width, height, fps, frame_count

In [4]:
example = paths[animals[0]]['pairs'][0]

extract_video_info(example['video'])

(2560, 1440, 30.0, 284)

In [7]:
frames = load_video(ds_info['data']['horse'][0]['video'])
annotations = load_gt(ds_info['data']['horse'][0]['label'])
formated_annotations = format_gt(annotations)

print(frames[0].shape)
print(len(annotations))

(1440, 2560, 3)
6653


In [ ]:
for animal in animals:
    print(f'Animal: {animal}')
    for pair in paths[animal]['pairs']:
        info = extract_video_info(pair['video'])
        print(f'Resolution: {info[0]}x{info[1]}, FPS: {info[2]}, Frames: {info[3]}')

Animal: horse
Resolution: 2560x1440, FPS: 30.0, Frames: 284
Resolution: 2560x1440, FPS: 30.0, Frames: 310
Resolution: 2560x1440, FPS: 30.0, Frames: 1434
Resolution: 2560x1440, FPS: 30.0, Frames: 206
Resolution: 2560x1440, FPS: 30.0, Frames: 325
Resolution: 2560x1440, FPS: 30.0, Frames: 315
Resolution: 2560x1440, FPS: 30.0, Frames: 1315
Animal: penguin
Resolution: 2560x1440, FPS: 30.0, Frames: 314
Resolution: 2560x1440, FPS: 30.0, Frames: 325
Resolution: 1920x1080, FPS: 30.0, Frames: 322
Resolution: 2560x1440, FPS: 30.0, Frames: 328
Resolution: 2560x1440, FPS: 30.0, Frames: 321
Resolution: 2560x1440, FPS: 30.0, Frames: 234
Animal: pig
Resolution: 2560x1440, FPS: 30.0, Frames: 315
Resolution: 2560x1440, FPS: 30.0, Frames: 312
Resolution: 2560x1440, FPS: 30.0, Frames: 312
Resolution: 2560x1440, FPS: 30.0, Frames: 274
Resolution: 2560x1440, FPS: 30.0, Frames: 318


In [ ]:
def draw_bbox(frame, ann):
    """
    Draw a bounding box on a frame.
    
    Args:
        frame (np.ndarray): Frame to draw on.
        ann (dict): Annotation to draw.
    
    Returns:
        np.ndarray: Frame with bounding box drawn.
    """
    x1 = int(ann["top_left_x"])
    y1 = int(ann["top_left_y"])
    x2 = int(ann["top_left_x"] + ann["width"])
    y2 = int(ann["top_left_y"] + ann["height"])
    target_id = ann["target_id"]
    confidence = ann["confidence"]
    class_id = ann["class_id"]
    visibility = ann["visibility"]
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(frame, f"ID: {target_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    cv2.putText(frame, f"Conf: {confidence:.2f}", (x1, y1 - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    cv2.putText(frame, f"Class: {class_id}", (x1, y1 - 50), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    cv2.putText(frame, f"Vis: {visibility:.2f}", (x1, y1 - 70), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    return frame

In [ ]:
def save_video(frames, save_path, fps, width, height):
    """
    Save a list of frames as a video.
    
    Args:
        frames (list): List of frames to save.
        save_path (str): Path to save the video.
    
    Returns:
        pathlib.Path: Path to the saved video.
    """
   
    video = cv2.VideoWriter(str(save_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    for frame in frames:
        video.write(frame)
    video.release()
    return save_path
    
    

In [ ]:

def get_video_with_annotations(video_path, label_path, save_path):
    """
    Plot a video with annotations.
    
    Args:
        video_path (str): Path to the video file.
        label_path (str): Path to the label file. Where label is in format:
            [frame_id, targer_id, top_left_x, top_left_y, width, height, confidence, class, visibility]
        save_path (str): Path to save the video with annotations.
    
    Returns:
        pathlib.Path: Path to the saved video with annotations.
    """
    frames = load_video(video_path)
    labels = load_gt(label_path)
    formated_annotations = format_gt(labels)

    width, height, fps, frame_count = extract_video_info(video_path)
    
    for frame_id, frame in enumerate(frames):
        frames_ann = [a for a in formated_annotations if a["frame_id"] == frame_id]
        
        for ann in frames_ann:
            frame = draw_bbox(frame, ann)
    
    save_video(frames, save_path, fps, width, height)
    return save_path

In [ ]:
video_path = paths['horse']['pairs'][0]['video']
label_path = paths['horse']['pairs'][0]['label']
get_video_with_annotations(video_path, label_path, 'videp.mp4')

'videp.mp4'